# 02 Silver Layer
**CSE 487 Phase 3, Team CTRL-Freaks** (Silver layer by Josh Hu)

**Purpose:** Clean the raw Bronze data into one table that's ready for analysis. these are the same 8 cleaning steps from Phase 1 and Phase 2, js written with Spark instead of pandas, so our Phase 3 models train on the same rows as Phase 2.

**Inputs:** the Bronze Delta tables from `01_bronze_layer`
- `workspace.ctrl_freaks.bronze_311_old` (July 2008 - May 2024, every value is text)
- `workspace.ctrl_freaks.bronze_311_new` (May 2024 - Nov 19 2026 snapshot, evry value is text)

**Output:** Delta table `workspace.ctrl_freaks.silver_311`, one row per real 311 request from Jan 2020 on

**Assumptions:** `01_bronze_layer` already ran. Everything is Spark DataFrames, nothing gets turned into pandas.

**How to run:** Run all, top to bottom. It takes a few minutes bc every count makes Spark redo the steps above it.

| # | Operation | Why it runs here |
|---|---|---|
| 1 | Data Integration | Combines both datasets so every later step runs on one table |
| 2 | Removing Irrelevant Data | Trims rows and columns before any heavier processing |
| 3 | Data Type Conversion | Dates and coordinates must be real types before they can be checked |
| 4 | String Cleaning | Cleans text so later grouping and duplicate checks match correctly |
| 5 | Fixing Swapped & Placeholder Coordinates | Creates new missing values that Operation 6 then handles |
| 6 | Handling Missing Data | Deals with all missing values, including those from Operation 5 |
| 7 | Removing Duplicates | Uses the original request types, so it runs before they are grouped |
| 8 | Category Standardization | Groups request types into broader categories last |

## Setup
Read the two Bronze tables. I also keep a list of the row count after every step that removes rows, so we get a befor/after table at the end

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# same names as bronze, change here if ur catalog is different
catalogName = "workspace"
schemaName = "ctrl_freaks"

oldBronze = spark.table(f"{catalogName}.{schemaName}.bronze_311_old")
newBronze = spark.table(f"{catalogName}.{schemaName}.bronze_311_new")

# (step, rows) after evry step that drops rows
rowCounts = []
rowCounts.append(("Bronze old + new (all years)", oldBronze.count() + newBronze.count()))
print("bronze old:", oldBronze.count(), "| bronze new:", newBronze.count())

## Operation 1: Data Integration
**What**: Merge the two 311 datasets into one dataset covering January 2020 to present

**Why**: Our time range is split across two files, and the old and new city systems store the same info under diffrent column names.

**How**:
1. Filter the old data to requests opened on or after January 1, 2020
2. Rename the newer dataset's columns to match the older dataset (ex: `CreatedDate` → `OpenDate`, `CaseNumber` → `CaseReference`)
3. Keep only the columns both datasets share, plus `DuplicateRequest` (new data) and `CouncilDistrict2011` (old data), which later operations need
4. Add a `Source` column so each row can be traced back to its orignal file
5. Stack the two datasets with `unionByName`

Bronze's `sourceFile` and `ingestedAt` columns dont get kept bc `Source` does teh same job here.

In [ ]:
#step 1: only old rows from 2020 on
# i parse the date js to filter, the real date conversion is op 3
dateFormat = "MM/dd/yyyy hh:mm:ss a"
oldOpen = F.try_to_timestamp(F.col("OpenDate"), F.lit(dateFormat))
oldDf = oldBronze.filter(oldOpen >= F.lit("2020-01-01").cast("timestamp"))
print("old rows from 2020 on:", oldDf.count())

In [ ]:
#step 2: rename the new columns to the old names (left = new name, right = old name)
# PropertyID and PoliceDistrict alredy match once bronze took the spaces out
renameMap = {
    "CreatedDate": "OpenDate",
    "CaseNumber": "CaseReference",
    "Department": "Subject",
    "AssessedNumber": "AddressNumber",
    "AssessedStreet": "AddressLine1",
    "Zip": "Zipcode",
    "PlanningNeighborhood": "Neighborhood",
}
newDf = newBronze
for fromName, toName in renameMap.items():
    newDf = newDf.withColumnRenamed(fromName, toName)

In [ ]:
# step 3: keep only the columns both files have
sharedCols = [
    "CaseReference",
    "OpenDate",
    "ClosedDate",
    "Status",
    "Subject",
    "Type",
    "AddressNumber",
    "AddressLine1",
    "City",
    "State",
    "Zipcode",
    "PropertyID",
    "Latitude",
    "Longitude",
    "Location",
    "CouncilDistrict",
    "PoliceDistrict",
    "CensusTract",
    "CensusBlockGroup",
    "CensusBlock",
    "Neighborhood",
]
# old also keeps CouncilDistrict2011 (op 6 uses it), new keeps DuplicateRequest (op 7 uses it)
oldDf = oldDf.select(sharedCols + ["CouncilDistrict2011"])
newDf = newDf.select(sharedCols + ["DuplicateRequest"])

In [ ]:
# step 4: tag where each row came from
oldSource = "July 2008-May 2024 dataset"
newSource = "May 2024-Present dataset"
oldDf = oldDf.withColumn("Source", F.lit(oldSource))
newDf = newDf.withColumn("Source", F.lit(newSource))

# step 5: stack them, allowMissingColumns fills the column the other file doesnt have with null
df = oldDf.unionByName(newDf, allowMissingColumns=True)
mergedRows = df.count()
rowCounts.append(("Op 1: old filtered to 2020+, merged with new", mergedRows))
print("merged rows:", mergedRows, "| columns:", len(df.columns))

## Operation 2: Removing Irrelevant Data
**What:** remove rows and columns that dont help answer our research question.

**Why:** Test entries arent real requests, and extra columns add clutter wihtout adding value.

**How:**
- **Rows:** Drop the test entries where the department (`Subject`) is `"Test"`. I use `eqNullSafe` so rows with no Subject dont get dropped by accident (they get labeled Unknown in Op 4).
- **Columns:** Drop columns that repeat other info or arent needed:
  - `Location`: same info as `Latitude` and `Longitude` in a diffrent format
  - `City`, `State`: every row is Buffalo, NY
  - `CensusTract`, `CensusBlockGroup`, `CensusBlock`: census areas, our anaylsis compares council districts and neighborhoods instead
  - `PropertyID`: parcel IDs arent needed for our question

**Kept on purpose:** `CouncilDistrict2011` (used in Op 6) and `DuplicateRequest` (used in Op 7).

In [ ]:
isTest = F.col("Subject").eqNullSafe("Test")
print("test rows removed:", df.filter(isTest).count())
df = df.filter(~isTest)

dropCols = ["Location", "CensusTract", "CensusBlockGroup", "CensusBlock", "City", "State", "PropertyID"]
df = df.drop(*dropCols)

rowCounts.append(("Op 2: test rows removed", df.count()))
print("columns left:", df.columns)

## Operation 3: Data Type Conversion
What: Convert columns to thier correct data types.
Why: dates stored as text cant be used to calculate response times, whcih is the core of our project.

How:
- `OpenDate`, `ClosedDate` → timestamp (both files use the format `09/03/2026 02:15:00 PM`)
- `Latitude`, `Longitude` → double
- `DuplicateRequest` → true/false (old rows are blank bc the old system didnt have this flag, so they become false)

the `try_` functions give null when somthing cant be converted instead of crashing (same idea as `errors="coerce"` in pandas), and we count how many that happened to.

In [ ]:
print("types before (all text from bronze):")
df.printSchema()

In [ ]:
df = df.withColumn("OpenDate", F.try_to_timestamp(F.col("OpenDate"), F.lit(dateFormat)))
df = df.withColumn("ClosedDate", F.try_to_timestamp(F.col("ClosedDate"), F.lit(dateFormat)))
df = df.withColumn("Latitude", F.expr("try_cast(Latitude as double)"))
df = df.withColumn("Longitude", F.expr("try_cast(Longitude as double)"))

#old rows are null here (no duplicate flag in the old system), so they become false
df = df.withColumn("DuplicateRequest", F.coalesce(F.lower(F.col("DuplicateRequest")) == "true", F.lit(False)))

print("types after:")
df.printSchema()

# open date should have 0 fails. no closed date means the case is still open, not a bad date
print("open dates that failed to parse:", df.filter(F.col("OpenDate").isNull()).count())
print("closed date empty (open cases):", df.filter(F.col("ClosedDate").isNull()).count())
display(df.select(F.min("OpenDate").alias("earliest"), F.max("OpenDate").alias("latest")))

## Operation 4: String Cleaning
**What:** Clean up inconsistent text formatting and spelling in the `Type` and `Subject` (department) columns
**Why:** the same departmnet appears under different names (ex: `Parking` and `Dept of Parking`), which would split its requests into seperate groups.

**How:**
- Remove the `(Req_Serv)` suffix from request types, bc every row is alredy a service request
- Combine department name variants into one consistant name
- Label mising departments as `Unknown`

In [ ]:
# sql replace() swaps plain text, so no regex escaping needed for the ( )
df = df.withColumn("Type", F.expr("replace(Type, ' (Req_Serv)', '')"))

In [ ]:
# same department map as phase 2
deptMap = {"Mayor's Office": "Office of the Mayor",
           "Parking": "Parking Department",
           "Dept of Parking": "Parking Department",
           "Dept of Public Works": "Public Works, Parks, & Streets Department",
           "Public Works, Parks & Streets": "Public Works, Parks, & Streets Department",
           "Community Services & Rec. Program.": "Community Service & Recreational Programming",
           "Community Services & Recreational Programming": "Community Service & Recreational Programming",
           "DPIS": "Permit & Inspection Services",
           "Dept of Law": "Law Department"
}
df = df.replace(deptMap, subset=["Subject"])
df = df.fillna({"Subject": "Unknown"})

## Operation 5: Fixing Swapped & Placeholder Coordinates
**What:** correct invalid values in `Latitude` and `Longitude`.
**Why:** Bad coordinates would place requests in the wrong location on maps and in any location-based analysis.

**How:**
1. **Swapped coordinates (new data):** Buffalo's latitude is positive (about 42.9), so any row with a negative latitude was flipped, and we swap it back. I save a flag and the old latitude first, otherwise the second line would read the latitude that already got swapped.
2. **Placeholder cordinates (old data):** Most old rows from 2020 to 2023 sit at exaclty (43.0, -79.0). thats a default the old system filled in, not a real spot, so we set these to missing. these rows still have a council district and neighborhod, so they stay useable
3. **Boundary check:** Any remaining coordinates outside Buffalo (latitude 42.82 to 42.97, longitude -78.92 to -78.79) are set to missing

*this runs befor Operation 6 bc it creates new missing values that Operation 6 then handles.*

In [ ]:
# negative latitude = lat and lon got flipped
df = df.withColumn("isSwapped", F.coalesce(F.col("Latitude") < 0, F.lit(False)))
print("rows with swapped coordinates:", df.filter(F.col("isSwapped")).count())
display(df.filter(F.col("isSwapped")).groupBy("Source").count())

df = df.withColumn("tempLat", F.col("Latitude"))
df = df.withColumn("Latitude", F.when(F.col("isSwapped"), F.col("Longitude")).otherwise(F.col("Latitude")))
df = df.withColumn("Longitude", F.when(F.col("isSwapped"), F.col("tempLat")).otherwise(F.col("Longitude")))
df = df.drop("isSwapped", "tempLat")

print("negative latitude after fix:", df.filter(F.col("Latitude") < 0).count())

In [ ]:
#(43.0, -79.0) shows up 300k+ times, its a fake default spot
df = df.withColumn("isPlaceholder", F.coalesce((F.col("Latitude") == 43) & (F.col("Longitude") == -79), F.lit(False)))
print("placeholder coordinate rows:", df.filter(F.col("isPlaceholder")).count())

df = df.withColumn("Latitude", F.when(~F.col("isPlaceholder"), F.col("Latitude")))
df = df.withColumn("Longitude", F.when(~F.col("isPlaceholder"), F.col("Longitude")))
df = df.drop("isPlaceholder")

In [ ]:
# anything outside buffalo is treated as a bad coordinate
inBuffalo = F.col("Latitude").between(42.82, 42.97) & F.col("Longitude").between(-78.92, -78.79)
df = df.withColumn("isOutside", F.col("Latitude").isNotNull() & ~F.coalesce(inBuffalo, F.lit(False)))
print("coordinates outside buffalo:", df.filter(F.col("isOutside")).count())

df = df.withColumn("Latitude", F.when(~F.col("isOutside"), F.col("Latitude")))
df = df.withColumn("Longitude", F.when(~F.col("isOutside"), F.col("Longitude")))
df = df.drop("isOutside")

print("rows with valid coordinates:", df.filter(F.col("Latitude").isNotNull()).count(), "/", df.count())

## Operation 6: Handling Missing Data
**What**: Find all missing values and decide how to handle each column.

**Why**: both city systems write the text `"UNKNOWN"` (and sometimes `"UnAssigned"`) instead of leaving cells blank, and the old file writes `"NULL"` in a few address cells, so a normal null check undercounts them. (pandas turned `NULL` into missing on its own when it read the CSV, but Spark keeps it as text, so we do it here.)

**How**:
1. Convert `"UNKNOWN"`, `"UnAssigned"`, `"NULL"`, and empty text to real nulls, then compare missing counts before and after. this happns before step 2 so `CouncilDistrict2011` gets cleaned too.
2. **`CouncilDistrict`:** the old system left this blank for most of 2020 to 2021. `CouncilDistrict2011` is filled for most of those rows, so we use it to fill the gaps n then drop it. *Limitation: 2011 district lines dont perfectly match current ones, so a small share of filled-in districts may be off.*
3. **Rows still missing `CouncilDistrict`:** Dropped, bc comparng districts is the point of our project
4. **Missing `ClosedDate`:** these are still-open cases, not bad data. we keep them and flag them with a new `isOpen` column so response time analysis can skip them
5. **`Latitude`, `Longitude`, `Zipcode`, `Neighborhood`, `PoliceDistrict`, address columns:** Left as missing, since not every analysis needs them and dropping those rows would throw away useful data

In [ ]:
# these 4 columns are where the city writes UNKNOWN insted of leaving it blank
checkCols = ["CouncilDistrict", "Zipcode", "Neighborhood", "PoliceDistrict"]
for colName in checkCols:
    print(colName, "missing before:", df.filter(F.col(colName).isNull()).count())

# only text columns can say UNKNOWN or NULL
textCols = []
for colName, colType in df.dtypes:
    if colType == "string":
        textCols.append(colName)

#UNKNOWN / UnAssigned / NULL / empty -> real null
for colName in textCols:
    df = df.withColumn(colName, F.when(F.col(colName).isin("UNKNOWN", "UnAssigned", "NULL", ""), None).otherwise(F.col(colName)))

for colName in checkCols:
    print(colName, "missing after:", df.filter(F.col(colName).isNull()).count())

In [ ]:
# fill mising council districts from the 2011 colum, then drop it
print("council district missing before:", df.filter(F.col("CouncilDistrict").isNull()).count())
df = df.withColumn("CouncilDistrict", F.coalesce(F.col("CouncilDistrict"), F.col("CouncilDistrict2011")))
print("council district missing after fill:", df.filter(F.col("CouncilDistrict").isNull()).count())
df = df.drop("CouncilDistrict2011")

In [ ]:
# no district = cant compare it, so drop
print("rows with no district dropped:", df.filter(F.col("CouncilDistrict").isNull()).count())
df = df.filter(F.col("CouncilDistrict").isNotNull())

# keep open cases but flag them so response time stuff can skip them
df = df.withColumn("isOpen", F.col("ClosedDate").isNull())
print("open cases kept:", df.filter(F.col("isOpen")).count())

rowCounts.append(("Op 6: rows with no council district removed", df.count()))
print("rows left:", df.count())

## Operation 7: Removing Duplicates
**What**: remove repeat reports of the same issue.

**Why**: Duplicates inflate request counts for some areas, which would skew comparisons between districts.

**How**:
1. Check for exact duplicate rows and repeated case IDs (both should be 0, and the two datasets' date ranges dont overlap)
2. Remove requests the new system flagged itself as duplicates (`DuplicateRequest = true`)
3. Remove near-duplicates: same request type, same address, same day, keeping the earliest report. Rows w no street address are skipped, since we cant confirm they're teh same location. In Spark this is a window: number the rows in each (type, address, day) group from earliest to latest and keep number 1. If two have the exact same time, the lower case number wins so the result is the same every run.

*this runs before Operation 8 so near-duplicates are matched on the original, specific request types.*

In [ ]:
totalRows = df.count()
print("exact duplicate rows:", totalRows - df.dropDuplicates().count())
print("duplicate case ids:", totalRows - df.dropDuplicates(["CaseReference"]).count())

In [ ]:
# the city's new system flags its own duplciates
print("flagged duplicates removed:", df.filter(F.col("DuplicateRequest")).count())
df = df.filter(~F.col("DuplicateRequest"))
df = df.drop("DuplicateRequest")
rowCounts.append(("Op 7: flagged duplicates removed", df.count()))

In [ ]:
#same type + same adress + same day = same issue reported again, keep the earliest one
df = df.withColumn("createdDay", F.to_date(F.col("OpenDate")))
dupWindow = Window.partitionBy("Type", "AddressNumber", "AddressLine1", "createdDay").orderBy("OpenDate", "CaseReference")
df = df.withColumn("dupNumber", F.row_number().over(dupWindow))

# skip rows w no street bc we cant tell if its the same place
isNearDupe = (F.col("dupNumber") > 1) & F.col("AddressLine1").isNotNull()
print("near-duplicates removed:", df.filter(isNearDupe).count())
display(df.filter(isNearDupe).groupBy("Source").count())

df = df.filter(~isNearDupe)
df = df.drop("createdDay", "dupNumber")
rowCounts.append(("Op 7: near-duplicates removed", df.count()))
print("rows left:", df.count())

## Operation 8: Category Standardization
What: group 300+ specfic reqeust types into broader categories
Why: Many reqeust types describe the same issue under different names (ex: `Pot Hole`, `Pothole Issue`, and `Other Hole in Road`), whcih makes comparisons by type unreliable.

How: each request type is mapped into one of 24 categories (ex: `Roads & Potholes`, `Snow Management`). Anything that doesnt match a category is labeled `Other`. the lists are copied exactly from Phase 2, typos included, bc they have to match the city's spelling.

In [ ]:
print("unique request types before grouping:", df.select("Type").distinct().count())

In [ ]:
#Waste categories
totes_bins = ['Totes Audit', 'Totes Deliver', 'Tote Pickup', 'Totes Pickup',
              'Totes Replace', 'Totes Combo', 'Totes Abandon Pickup',
              'Totes Report of Removed Broken Tote', 'Special Even Totes',
              'Order Additional or Change Current Totes', 'Replacement Garbage Totes Request',
              'Recycling Tote Replace', 'Recycyling Tote Deliver', 'Recycling Tote Pickup',
              'Recycling Tote Abandon Pickup', 'Recycling Tote Combo', 'Recycling Tote Combo (Exchange Size)','Impact Bill Inquiry or Concern',
              'Special Event Totes']
regular_bulk_collection = [
    "Garbage Missed Pick Up", "Recycling Missed Pick Up", "Missed Pickup 2_Piece Large Trash",
    "Missed Pickup; 2 Piece Large Trash", "Missed Pickup; 3 Piece Large Trash",
    "Bulk Trash", "Bulk Trash Issue", "Scrap_It Curbside",
    "Senior or Disabled Citizen, TV Pickup Request",
    "Senior or Disabled Citizen, TV/Electronics Pickup Request", "Electronic Waste",
    "TV and Electronic Waste at Curb", "Fridge", "Fridge Removal", "Christmas Tree",
    "Christmas Tree Removal", "Leaves / Lawn Debris", "High Grass/Weeds",
    "Shopping Carts (Impact Team)", 'Trash Ordinance Violation','Rain Barrels'
]
illegal_dumping_debris = [
    "Excess Trash", "Illegal Dumping Curb", "Illegal Dumping PrivateProperty",
    "Illegal Dumping", "OSP Illegal Dumping", "QRT Illegal Dumping on Viaduct",
    "Illegal Dumping Private Property / Vacant Lot", "Great American Clean-Up",
    "Neighborhood CleanUp", "Neighborhood CleanUp Request",
    "Weather Emergency Trash Pick Up - 9/3/2026 Event", 'Illegal Dumping Street'
]

In [ ]:
# Streets, sidewalks, and roads
roads_and_potholes = [
    "Pot Hole", "Other Hole in Road", "Pothole Issue", "Paving", "Street Paving Request",
    "Cave In", "PW Ongoing Construction", "PW Ongoing Construction Issue",
    "PW Engineering", "Obscene PW Engineering", 'Engineering Operational', 'Damage from Street Worker',
    'CityHall_CityCourt Maintenance'
]
sidewalks_and_curbs = [
    "Sidewalks", "New Sidewalk/Ramp", "Sidewalk(s) Repair Request", "Sidewalk(s) Issue",
    "Curbs", "Curbs Replacement Request", "Curb - Metal Protruding",
    "Curb - Metal Protruding Issue", "Bike Rack", "Bike Rack Issue", "Bike Rack Request"
]
lighting_and_signals = [
    "Streetlights", "Streetlight(s) Issues", "Damaged Street Light Pole",
    "Damaged or Missing Street Light Pole", "Signal Out or Flashing", "Signal Issue DoT",
    "Signal Other Issue", "Signal Issue", "Signal Timing Issue City",
    "Signal Timing Issue in City of Buffalo"
]
traffic_and_lines = [
    "Pavement Marking Lines", "Pavement Marking Issue", "Buffalo Traffic Violations",
    "PVB School-Zone Camera Citations", "PW Traffic", "Obscene PW Traffic",
    "PW Neighborhood Traffic Calming", "Neighborhood Traffic Calming", "Speeding Issue(s)",
    "Bridge Issue", "Report Bridge Flooding", "Weather Event Damage to ROW Infrastructure",
    "Report Snow on Bridge or Viaduct", "Sweeper", "Sweeper Request", "Seweeper Issue", "ROW Bike Lane ; on-road or cycle tracks Issue"

]
obstructions_and_hazards = [
    "Basketball Hoop in RoW", "Basketball Hoop in Street", "Right of Way Issue",
    "Sign Hazards", "Sign Maintenance", "Sign Issue", "Down Wire(s)"]

In [ ]:
# Forests and parks
tree_maintenance = [ "Tree Removal",
    "Tree Removal (Req_SERV)", "Tree Trimming Request", "Fallen Tree Blocking RoW",
    "Tree Other", "Fallen Tree Inspection", "Tree Removal Quality Issue (Req_serv)",
    "Tree Removal Challenge", "Stump Removal", "Tree Planting Request",
    "Tree Removal per Inspector", "Tree Pruning Request", "Tree Pruning Private Property",
    "Tree Planting Quality Issue", "Stump Removal Quality Issue",
    "Tree Trimming Quality Issue", "City Park Tree Issue",
    "Fallen Tree, split tree, hanging limb",
    "Fallen Tree Inspection on Private Property"
]
parks_and_recreation = [
    "City Parks", "City Parks Issue", "Parks City", "Olmsted Parks", "Olmsted Parks Issue",
    "Parks Olmsted", "RalphWilson Park Construction", "Ralph Wilson Park Issue",
    "Pools and Splashpads", "Pools and Splashpads Issue", "Recreation Center",
    "Recreation Center Issue", "Youth Bureau Issue", "Buffalo Place Park Issue",
    "Naval Park Issue", "Erie Basin Marina Issue",
    "Shoreline Trail, North Buffalo Rails to Trails, Jesse Kregal Pathway Issue",
    "Circles Medians Triangles Issues (Contractor)", "Obscene Parks City",
    "Obscene Parks Olmsted", "Parks Buildings Maintenance",
    "Tifft Nature Preserve Issue  (Buffalo Museum of Science)",
    "Pat Sole Park (Peace Bridge Authority)", "Ice Rink Issue"]

In [ ]:
water_services = [
    "Water Issue", "Water_Billing_Meter", "Buffalo Water Service Request",
    "Buffalo Water Service Issue", "Buffalo Water Billing Request",
    "Buffalo Water Testing Request", "Water Tested"
]
drainage_and_sewers = [
    "Sewer", "Sewer Issue", "Report Flooding", "Street Flooding",
    "Basement Flooding", "Basement Flooding Issue", "PW Missing Manhole Cover"
]

In [ ]:
# Housing and building code violations
violations_and_inspections = [
    "Housing Violations", "Open311 Housing", "Rental Registration",
    "Healthy Homes / PRI Inspection", "Housing or Building Violations(Exterior)",
    "Housing or Building Violations(Interior)", "Electrical Issue",
    "Electrical Violation (Interior)", "Electrical Violation (Exterior)",
    "Heating or A/C Issues(Exterior)", "Heating or A/C Issues(Interior)",
    "Property Issue(s)  (Quality of Life Issue)", "Short Term Rental Issue",
    "Zoning Board of Appeals Concern", 'Lead Paint Inspection', 'Building Maintenance', 'License Concern','Rodents',
    'Rodent Control Issue'
]
vacant_and_abandoned_properties = [
    "PW Vacant Lot", "Vacant Lot Cutting", "Inrem Real Estate", "Real Estate Issue",
    "Boarding request", "Board Up request", "Abandoned Property",
    "BPD Request Board Up/Cleanup Task, For Impact Team"
]
graffiti_and_obscenity = [
    "Open311 Graffiti", "Graffiti issue", "Graffiti Private Property",
    "Obscene City Property", "Obscene Parks City", "Obscene Private Property",
    "Obscene Other", "Graffiti, City Property or Right of Way (Impact Team)"
]

In [ ]:
# Parking & vehicles
parking_enforcement_and_rules = [
    "Parking Issues", "General Parking Issue(s)",
    "Illegal Parking (Quality of Life Issue)", "Unlicensed Street Vendor",
    "Parking Pay Station Issue(s)", "Parking Mobile App Issue(s)",
    "Parking Single Meter Issue(s)"]
meters_and_apps = [
    "User Fee", "Public Works User Fee Issue", "PVB Mobile App Issue",
    "PVB Mobile App Ticket", "PVB Single Meter", "PVB Pay Station"
]
abandoned_vehicles = [
    "Abandoned Vehicles", "Abandoned Vehicle Inspection",
    "Privately Owned Property, Abandoned Vehicle", "Unregistered Vehicle",
    "Abandoned Vehicles (Parking Dept)",
    "Vehicle Immobilized in Street (Parking Dept)"]

In [ ]:
# Public safety, fire, and animals
animal_and_wildlife_control = [
    "Animals", "Dead Animal Removal", "Dead Animal Removal Request",
    "Dog License Issue", "Dog Animal Control Issue", "Wildlife Control Issue",
    "Bees and Hornets Animal Control Issue", "Personal Wildlife Trap Report",
    "Pest", "Pest_Private Trap", "Chicken Coop", "Chicken Issue",
    "Dog Licensing Issue", "Livestock Issue"
]
nuisance_and_quality_of_life = [
    "Excessive Noise", "Noise Nuisance (Loud Music)", "Loitering", "Panhandling",
    "Homeless Concern (Quality of Life Issue)", "Juvenile / Youth Concern",
    "Quality of Life Issue", "Quality of Life Issue (Other)", "Good Neighbor",
    "Sneakers Hanging", "ATV and or Dirt Bike", "Unlicensed Business"
]
fire_safety_and_police = [
    "Fire", "BFD Fire Prevention", "Buffalo Fire Prevention Request",
    "General Fire Issue", "Police Issue", "Police Issue (Other)",
    "Public Safety Support - (BFD)"
]

In [ ]:
# Winter/weather
snow_management = [
    "Street Snow Plowing Issue", "Street Snow Plowing", "Street Snow Plowing Request",
    "Street Salting", "Street Salting Issue", "Street Salting Request",
    "QRT Snow Removal", "Snow Removal", "Snow Removal Inspection",
    "Sidewalk Snow Removal Inspection", "Sidewalk Snow/Ice Removal Inspection",
    "Snow Removal Bus Stop/Shelter", "Damage from Snow Removal",
    "Vehicle Blocking Snow Plow", "Snow Removal Concern", "Snow-Bank Removal",
    "BFD Snow on Hydrant", "Buffalo Fire Department Snow on Hydrant"]

In [ ]:
#Internal or administrative issues
administrative_and_records = [
    "FOIL Records EDPIS", "FOIL Records Citizens Services",
    "FOIL Records Board of Ethics", "City Clerk Issue", "General City Clerk Issue",
    "Taxation Issue", "Assessment Issue", "2020 Reassessment",
    "Compensation & Benefits", "New Americans Request"
]

internal_and_operations = [
    "BMHA Issue", "Pick and Pay", "Open311 APP Issues",
    "Report Technical Issue for Buff311 App", "Telecommunications",
    "General Impact Team Request", "Johnnie B Wiley Issue (BPS)", "Schools",
    "Save Our Streets Program", "Other Adjudication Issue",
    "Outer Harbor Parkway (Empire State Development Corp)", "CBO",
    "Fair Housing Issue", "FairHsg Other", "Security Deposit",
    "Union Ship Canal Issue (BUDC)", "DPIS Department Operational Complaint",
    "Obscene Schools", "DPIS Personnel Complaint"
]

In [ ]:
# Map each category name to its list of request types
mapping = {"Totes & Bins" : totes_bins,
           "Regular & Bulk Collection":regular_bulk_collection,
           "Illegal Dumping & Debris" : illegal_dumping_debris,
           "Roads & Potholes" : roads_and_potholes,
           "Sidewalks & Curbs": sidewalks_and_curbs,
           "Lighting & Signals" : lighting_and_signals,
           "Traffic & Lines": traffic_and_lines,
           "Obstruction & Hazards": obstructions_and_hazards,
           "Tree Maintenance": tree_maintenance,
           "Parks & Recreation":parks_and_recreation,
           "Water Services": water_services,
           "Drainage & Sewers": drainage_and_sewers,
           "Violations & Inspections": violations_and_inspections,
           "Vacant & Abandoned Properties": vacant_and_abandoned_properties,
           "Graffiti & Obscenity": graffiti_and_obscenity,
           "Parking Enforcement & Rules": parking_enforcement_and_rules,
           "Meters & Apps": meters_and_apps,
           "Abandoned Vehicles": abandoned_vehicles,
           "Animal & Wildlife Control" : animal_and_wildlife_control,
           "Nuisance & Quality of Life": nuisance_and_quality_of_life,
           "Fire Safety & Police" : fire_safety_and_police,
           "Snow Management": snow_management,
           "Administrative & Records": administrative_and_records,
           "Internal & Operations" : internal_and_operations
           }

In [ ]:
#turn the lists into one type -> category lookup
# "Obscene Parks City" is in 2 lists, first one wins (same as phase 2) so the join cant double any rows
typeToGroup = {}
for group, lst in mapping.items():
    for typeName in lst:
        if typeName not in typeToGroup:
            typeToGroup[typeName] = group

lookupDf = spark.createDataFrame(list(typeToGroup.items()), ["Type", "category"])

rowsBefore = df.count()
df = df.join(lookupDf, on="Type", how="left")
df = df.withColumn("Type", F.coalesce(F.col("category"), F.lit("Other")))
df = df.drop("category")
assert df.count() == rowsBefore, "join changed the row count"

print("unique request types after grouping:", df.select("Type").distinct().count())
display(df.groupBy("Type").count().orderBy(F.desc("count")))

## Final Checks
Each check confirms one operation did what it was supposed to. If any check fails, the notebook stops with an error.

In [ ]:
assert df.filter(F.col("Latitude") < 0).count() == 0
assert df.filter(F.col("Latitude").isNotNull() & ~F.col("Latitude").between(42.82, 42.97)).count() == 0

unknownLeft = 0
for colName, colType in df.dtypes:
    if colType == "string":
        unknownLeft = unknownLeft + df.filter(F.col(colName).isin("UNKNOWN", "NULL")).count()
assert unknownLeft == 0

assert df.filter(F.col("CouncilDistrict").isNull()).count() == 0
assert dict(df.dtypes)["OpenDate"] == "timestamp"
assert dict(df.dtypes)["ClosedDate"] == "timestamp"
assert df.count() == df.dropDuplicates(["CaseReference"]).count()

print("all checks passed!")
display(df.groupBy("Source").count())

## Row Counts Before and After
every step that removes rows, w how many it removd

In [ ]:
countTable = []
prevRows = None
for step, rows in rowCounts:
    if prevRows is None:
        countTable.append((step, rows, 0))
    else:
        countTable.append((step, rows, prevRows - rows))
    prevRows = rows
display(spark.createDataFrame(countTable, ["step", "rows", "removed"]))

## Write the Silver Table
Same column order every time so Gold and MLlib know what they're getting. Note `Type` holds the category now (same as Phase 2)

In [ ]:
silverCols = [
    "CaseReference",
    "OpenDate",
    "ClosedDate",
    "isOpen",
    "Status",
    "Subject",
    "Type",
    "CouncilDistrict",
    "Neighborhood",
    "PoliceDistrict",
    "Zipcode",
    "AddressNumber",
    "AddressLine1",
    "Latitude",
    "Longitude",
    "Source",
]
df = df.select(silverCols)

silverTable = f"{catalogName}.{schemaName}.silver_311"
rowsBeforeWrite = df.count()

#overwrite so rerunning js replaces it, overwriteSchema in case we change a column later
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silverTable)
print("wrote", rowsBeforeWrite, "rows to", silverTable)

## Delta Checks
A write command alone doesnt prove anythign, so I read every table back from the catalog and check it.

In [ ]:
silverBack = spark.table(silverTable)
silverBackRows = silverBack.count()
if silverBackRows == rowsBeforeWrite:
    print("PASS: silver read back", silverBackRows, "rows, same as before the write")
else:
    print("FAIL: wrote", rowsBeforeWrite, "but read back", silverBackRows)
silverBack.printSchema()
display(silverBack.limit(5))

In [ ]:
# bronze and silver together, read back from the catalog
checkRows = []
for tableName in ["bronze_311_old", "bronze_311_new", "silver_311"]:
    fullName = f"{catalogName}.{schemaName}.{tableName}"
    back = spark.table(fullName)
    tableFormat = spark.sql(f"DESCRIBE DETAIL {fullName}").first()["format"]
    checkRows.append((fullName, back.count(), len(back.columns), tableFormat))
display(spark.createDataFrame(checkRows, ["table", "rows", "columns", "format"]))

## Summary
| Step | Rows after | Removed |
|---|---|---|
| Bronze old + new (all years) | 1,342,085 | |
| Op 1: old filtered to 2020+, merged w new | 575,349 | 766,736 (old rows before 2020) |
| Op 2: test rows removed | 575,345 | 4 |
| Op 6: no council district | 552,855 | 22,490 |
| Op 7: flagged duplicates | 547,842 | 5,013 |
| Op 7: near-duplicates | 533,940 | 13,902 |

**Values we changed but kept the row:**
- 10,126 swapped coordinates flipped back (all from the new data)
- 309,526 placeholder (43, -79) coordinates set to missing, 0 outside Buffalo
- 160,203 rows had no council district (155,534 of them said `UNKNOWN`), 137,713 got filled from the 2011 colum and the other 22,490 were dropped
- 14,850 open cases (no closed date) kept and flagged with `isOpen`
- 308 request types grouped into 24 categories + Other

**Check against Phase 2:** we ran the Phase 2 pandas cleaning on the same snapshot and every row count above matches, ending at the same 533,940 rows. the one thing that was different at first: the old file writes `NULL` in a few adress cells. pandas treats that as missing automatically, but Spark kept it as text, so Op 6 now converts it too

**For Gold and MLlib:** read `workspace.ctrl_freaks.silver_311`. `Type` is the category now (same as Phase 2), and the response time features (`resolutionDays`, `season`, `isSlow`, ...) still need to be made in Gold.